# Step 6 — Model Evaluation & Deployment

Evaluate all models on test set, compare metrics, select best model, save for production.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay, roc_curve)
import joblib

X_test  = pd.read_csv('../../data/processed/X_test.csv',  index_col=0)
y_test  = pd.read_csv('../../data/processed/y_test.csv',  index_col=0).squeeze()
trained_pipelines = joblib.load('../../model/all_pipelines.pkl')

print('X_test:', X_test.shape)
print('Models loaded:', list(trained_pipelines.keys()))

## 6.1 Evaluate All Models

In [ ]:
eval_results = {}

for name, pipe in trained_pipelines.items():
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    eval_results[name] = {
        'y_pred':  y_pred,
        'y_proba': y_proba,
        'acc': accuracy_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_proba)
    }

    print(f'\n=== {name} ===')
    print(f'Accuracy : {eval_results[name]["acc"]:.4f}')
    print(f'ROC-AUC  : {eval_results[name]["auc"]:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Good (0)', 'Bad (1)']))

## 6.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, eval_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Good', 'Bad']).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.suptitle('Confusion Matrices')
plt.tight_layout()
plt.show()

## 6.3 ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['steelblue', 'tomato', 'green']
for (name, res), color in zip(eval_results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color)
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6.4 Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': name, 'Accuracy': round(res['acc'], 4), 'ROC-AUC': round(res['auc'], 4)}
    for name, res in eval_results.items()
]).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print('=== Model Comparison ===')
print(summary.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.plot(x='Model', y='Accuracy', kind='bar', ax=axes[0], color='steelblue',
             edgecolor='black', legend=False)
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylim(0.5, 1.0)
axes[0].tick_params(axis='x', rotation=15)

summary.plot(x='Model', y='ROC-AUC', kind='bar', ax=axes[1], color='tomato',
             edgecolor='black', legend=False)
axes[1].set_title('ROC-AUC Comparison')
axes[1].set_ylim(0.5, 1.0)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 6.5 Feature Importance (Random Forest)

In [ ]:
NUM_FEATURES = ['Age', 'Job', 'Credit amount', 'Duration', 'Credit_per_Duration']
CAT_FEATURES = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Age_Group']

rf_pipe = trained_pipelines['Random Forest']
ohe_cols = rf_pipe.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(CAT_FEATURES)
feature_names = NUM_FEATURES + list(ohe_cols)
importances = rf_pipe.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 6.6 Select Best Model & Save for Production

In [ ]:
best_name = max(eval_results, key=lambda k: eval_results[k]['auc'])
best_pipeline = trained_pipelines[best_name]

print(f'Best Model : {best_name}')
print(f'Accuracy   : {eval_results[best_name]["acc"]:.4f}')
print(f'ROC-AUC    : {eval_results[best_name]["auc"]:.4f}')

joblib.dump(best_pipeline, '../../model/credit_risk_model.pkl')
print('\nBest model saved to model/credit_risk_model.pkl')

## 6.7 Verify Deployment — Sample Prediction

In [ ]:
loaded_model = joblib.load('../../model/credit_risk_model.pkl')

sample = pd.DataFrame([{
    'Age': 35, 'Sex': 'male', 'Job': 2, 'Housing': 'own',
    'Saving accounts': 'little', 'Checking account': 'moderate',
    'Credit amount': 5000, 'Duration': 24, 'Purpose': 'car',
    'Credit_per_Duration': 5000 / 24, 'Age_Group': 'Adult'
}])

pred  = loaded_model.predict(sample)[0]
proba = loaded_model.predict_proba(sample)[0][1]
label = 'HIGH RISK' if pred == 1 else 'LOW RISK'

print('=== Sample Prediction ===')
print(f'Prediction       : {pred}')
print(f'Risk Probability : {proba:.4f}')
print(f'Risk Label       : {label}')